# 02 Document Processing & Chunking

This notebook handles document loading, cleaning, and text chunking strategies (e.g., character, recursive character, semantic chunking) for the Hybrid RAG pipeline.

In [ ]:
# Import baseline libraries
import os
import sys

print("Document processing notebook initialized.")

In [ ]:
# ============================================================
# 02 - DOCUMENT PROCESSING & CHUNKING
# Hybrid RAG Project - PMAY-U
# ============================================================

import re
import json
from pathlib import Path

print("Libraries imported successfully")


# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_DIR = Path("/content/hybrid-rag-project")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("\nProject:", PROJECT_DIR)
print("Raw:", RAW_DIR)
print("Processed:", PROCESSED_DIR)


# ============================================================
# 2. LOAD RAW DOCUMENT
# ============================================================

file_path = RAW_DIR / "pmay_u.txt"

if not file_path.exists():
    raise FileNotFoundError(
        f"File not found: {file_path}\n"
        "Make sure pmay_u.txt exists in data/raw/"
    )

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

print("\nDocument loaded successfully")
print("Characters:", len(text))
print("Words:", len(text.split()))
print("Lines:", len(text.splitlines()))

print("\n--- DOCUMENT PREVIEW ---")
print(text[:1000])


# ============================================================
# 3. CLEAN DOCUMENT
# ============================================================

clean_text = text.replace("\r\n", "\n").replace("\r", "\n")

# Normalize spaces/tabs
clean_text = re.sub(r"[ \t]+", " ", clean_text)

# Remove excessive blank lines
clean_text = re.sub(r"\n{3,}", "\n\n", clean_text)

clean_text = clean_text.strip()

print("\nCleaning completed")
print("Original characters:", len(text))
print("Cleaned characters:", len(clean_text))


# ============================================================
# 4. SECTION EXTRACTION
# ============================================================

section_names = [
    "SCHEME NAME",
    "MINISTRY",
    "CATEGORIES",
    "DETAILS",
    "BENEFITS",
    "ELIGIBILITY",
    "APPLICATION PROCESS",
    "DOCUMENTS REQUIRED",
    "FREQUENTLY ASKED QUESTIONS",
    "SOURCE"
]

section_pattern = r"(?m)^(" + "|".join(
    re.escape(section) for section in section_names
) + r")\s*$"

matches = list(re.finditer(section_pattern, clean_text))

print("\nSections found:", len(matches))

for match in matches:
    print(" -", match.group(1))


# ============================================================
# 5. PARSE SECTIONS
# ============================================================

parsed_sections = {}

for i, match in enumerate(matches):

    section_name = match.group(1)

    start = match.end()

    if i + 1 < len(matches):
        end = matches[i + 1].start()
    else:
        end = len(clean_text)

    content = clean_text[start:end].strip()

    parsed_sections[section_name] = content


print("\nSection sizes:")

for section, content in parsed_sections.items():
    print(f"{section}: {len(content)} characters")


# ============================================================
# 6. CREATE STRUCTURED JSON
# ============================================================

processed_data = {
    "scheme_name": parsed_sections["SCHEME NAME"],
    "ministry": parsed_sections["MINISTRY"],
    "categories": [
        line.strip()
        for line in parsed_sections["CATEGORIES"].splitlines()
        if line.strip()
    ],
    "details": parsed_sections["DETAILS"],
    "benefits": parsed_sections["BENEFITS"],
    "eligibility": parsed_sections["ELIGIBILITY"],
    "application_process": parsed_sections["APPLICATION PROCESS"],
    "documents_required": parsed_sections["DOCUMENTS REQUIRED"],
    "faqs": parsed_sections["FREQUENTLY ASKED QUESTIONS"],
    "source": parsed_sections["SOURCE"]
}

pmay_json_file = PROCESSED_DIR / "pmay_u.json"

with open(pmay_json_file, "w", encoding="utf-8") as f:
    json.dump(
        processed_data,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nSaved:", pmay_json_file)


# ============================================================
# 7. SENTENCE SPLITTING FUNCTION
# ============================================================

def split_into_sentences(text):
    """
    Split text into sentences while preserving the text content.
    """

    text = re.sub(r"\s+", " ", text).strip()

    sentences = re.split(
        r"(?<=[.!?])\s+(?=[A-Z0-9\"'])",
        text
    )

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]


# ============================================================
# 8. CREATE SENTENCE RECORDS
# ============================================================

sentence_records = []

sentence_id = 0

for section_name, content in parsed_sections.items():

    # SOURCE is metadata and does not need sentence retrieval
    if section_name == "SOURCE":
        continue

    sentences = split_into_sentences(content)

    for position, sentence in enumerate(sentences):

        record = {
            "sentence_id": sentence_id,
            "scheme": "Pradhan Mantri Awas Yojana - Urban",
            "section": section_name,
            "position": position,
            "text": sentence
        }

        sentence_records.append(record)

        sentence_id += 1

print("\nTotal sentences:", len(sentence_records))

print("\n--- FIRST 10 SENTENCE RECORDS ---")

for record in sentence_records[:10]:
    print(record)


# ============================================================
# 9. SAVE SENTENCE RECORDS
# ============================================================

sentence_file = PROCESSED_DIR / "sentence_records.json"

with open(sentence_file, "w", encoding="utf-8") as f:
    json.dump(
        sentence_records,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nSaved:", sentence_file)


# ============================================================
# 10. CREATE SENTENCE-WINDOW REPRESENTATION
# ============================================================

WINDOW_SIZE = 1

sentence_windows = []

# Group sentences by section for efficient lookup
section_sentence_map = {}

for record in sentence_records:

    section = record["section"]

    if section not in section_sentence_map:
        section_sentence_map[section] = []

    section_sentence_map[section].append(record)


for record in sentence_records:

    sentence_id = record["sentence_id"]
    section = record["section"]

    section_sentences = section_sentence_map[section]

    position = record["position"]

    start = max(0, position - WINDOW_SIZE)
    end = min(
        len(section_sentences),
        position + WINDOW_SIZE + 1
    )

    window_records = section_sentences[start:end]

    window_text = " ".join(
        r["text"]
        for r in window_records
    )

    sentence_windows.append({
        "sentence_id": sentence_id,
        "section": section,
        "target_sentence": record["text"],
        "window_text": window_text
    })


print("\nTotal sentence windows:", len(sentence_windows))

print("\n--- SAMPLE SENTENCE WINDOW ---")

for window in sentence_windows[:5]:
    print(window)
    print()


# ============================================================
# 11. SAVE SENTENCE WINDOWS
# ============================================================

window_file = PROCESSED_DIR / "sentence_windows.json"

with open(window_file, "w", encoding="utf-8") as f:
    json.dump(
        sentence_windows,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved:", window_file)


# ============================================================
# 12. CREATE PARENT-CHILD REPRESENTATION
# ============================================================

parent_child_records = []

parent_id = 0
child_id = 0

# Number of sentences in each child chunk
CHILD_SIZE = 3

for section_name, content in parsed_sections.items():

    # SOURCE is metadata
    if section_name == "SOURCE":
        continue

    sentences = split_into_sentences(content)

    # Entire section becomes the parent
    parent_text = content

    for i in range(0, len(sentences), CHILD_SIZE):

        child_sentences = sentences[
            i:i + CHILD_SIZE
        ]

        child_text = " ".join(
            child_sentences
        )

        parent_child_records.append({
            "parent_id": parent_id,
            "child_id": child_id,
            "section": section_name,
            "parent_text": parent_text,
            "child_text": child_text,
            "child_position": i
        })

        child_id += 1

    parent_id += 1


print("\nTotal parent-child records:",
      len(parent_child_records))

print("\n--- SAMPLE PARENT-CHILD RECORDS ---")

for record in parent_child_records[:3]:
    print(record)
    print()


# ============================================================
# 13. SAVE PARENT-CHILD RECORDS
# ============================================================

parent_child_file = (
    PROCESSED_DIR / "parent_child_records.json"
)

with open(parent_child_file, "w", encoding="utf-8") as f:
    json.dump(
        parent_child_records,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved:", parent_child_file)


# ============================================================
# 14. FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 60)
print("DOCUMENT PROCESSING COMPLETED")
print("=" * 60)

print("\nProcessed files:")

for file in PROCESSED_DIR.iterdir():
    print(" -", file.name)

print("\nStatistics:")
print(" - Total sentences:", len(sentence_records))
print(" - Sentence windows:", len(sentence_windows))
print(" - Parent-child records:", len(parent_child_records))

print("\nProcessing pipeline:")
print("Raw TXT")
print("   ↓")
print("Cleaning")
print("   ↓")
print("Section Extraction")
print("   ↓")
print("Sentence Records")
print("   ↓")
print("Sentence Windows")
print("   ↓")
print("Parent-Child Records")
print("   ↓")
print("JSON files")